# Stochastic Calculus, Step by Step · Lesson 18: The Itô toolbox

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Is it a martingale? Look at the dt term · 3. The exponential martingale · 4. Polynomial martingales · 5. The Itô product rule · 6. A preview of Girsanov: reweighting shifts the path · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 18,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. Is it a martingale? Look at the dt term

Write the process with Itô's lemma as $dX = a\,dt + b\,dW$. The $dW$ part is an Itô integral, a martingale (Lesson 15). So $X$ is a martingale exactly when the **drift $a$ is zero** (given that $b$ is well behaved). The test below groups 20,000 paths by $W_{0.5}$ and compares the average of $X_1$ in each group with $X_{0.5}$: a martingale shows no gap.

In [ ]:
rng = np.random.default_rng(seed)
n_paths = 200_000
W_h = rng.normal(0, math.sqrt(0.5), n_paths)                  # W at time 0.5
W_1 = W_h + rng.normal(0, math.sqrt(0.5), n_paths)            # W at time 1
cands = {
    "W³": (lambda w, t: w**3, "3W"),
    "W³ − 3tW": (lambda w, t: w**3 - 3 * t * w, "0"),
    "e^W": (lambda w, t: np.exp(w), "½e^W"),
    "e^(W − t/2)": (lambda w, t: np.exp(w - t / 2), "0"),
    "W⁴ − 6tW² + 3t²": (lambda w, t: w**4 - 6 * t * w**2 + 3 * t**2, "0"),
    "tW": (lambda w, t: t * w, "W"),
}
bins = np.quantile(W_h, np.linspace(0, 1, 11))
g = np.clip(np.digitize(W_h, bins[1:-1]), 0, 9)
test_rows = []
for name, (X, drift) in cands.items():
    now, later = X(W_h, 0.5), X(W_1, 1.0)
    gap = max(abs(later[g == k_].mean() - now[g == k_].mean()) for k_ in range(10))
    test_rows.append({"process": name, "drift": drift, "max_gap": float(gap), "martingale": drift == "0"})
out["test"] = {"rows": test_rows}
pd.DataFrame(test_rows)

## 3. The exponential martingale

For a constant $\theta$, $Z_t = e^{\theta W_t - \frac12\theta^2t}$. Itô's lemma with $f(t, w) = e^{\theta w - \frac12\theta^2t}$: $f_t = -\tfrac12\theta^2f$, $f_w = \theta f$, $f_{ww} = \theta^2f$, so

$$dZ = \big(-\tfrac12\theta^2 + \tfrac12\theta^2\big)Z\,dt + \theta Z\,dW = \theta Z\,dW .$$

No $dt$ term: a martingale, with $E[Z_t] = 1$. The $-\tfrac12\theta^2t$ is exactly what cancels the Itô correction.

In [ ]:
exp_rows = []
for th in (0.2, 0.5, 1.0, 2.0):
    Zv = np.exp(th * W_1 - 0.5 * th**2)
    exp_rows.append({"theta": th, "mean": float(Zv.mean()), "without_correction": float(np.exp(th * W_1).mean()),
                     "without_theory": math.exp(0.5 * th**2)})
out["expmart"] = {"rows": exp_rows}
pd.DataFrame(exp_rows)

## 4. Polynomial martingales

Lesson 13 proved $W_t^2 - t$ is a martingale by expanding squares. With Itô it is one line: $d(W^2 - t) = 2W\,dW + dt - dt = 2W\,dW$. The same way, $W^3 - 3tW$ and $W^4 - 6tW^2 + 3t^2$ have no $dt$ term (the test in Section 2 agrees). Their expectations stay at 0, which gives moments for free: $E[W_t^4] = 6tE[W_t^2] - 3t^2 = 3t^2$.

In [ ]:
out["poly"] = {"E_W3_3tW": float((W_1**3 - 3 * W_1).mean()), "E_hermite4": float((W_1**4 - 6 * W_1**2 + 3).mean())}
out["poly"]

## 5. The Itô product rule

For two Itô processes, $d(XY) = X\,dY + Y\,dX + dX\,dY$. The last term is new: it is $b_Xb_Y\,dt$ when both move with the same $W$. Example: the discounted share $D_t = e^{-rt}S_t$ with $dS = \mu S\,dt + \sigma S\,dW$:

$$dD = e^{-rt}dS - re^{-rt}S\,dt = (\mu - r)D\,dt + \sigma D\,dW .$$

It is a martingale exactly when $\mu = r$: the continuous-time version of Lesson 11.

In [ ]:
prod_rows = []
for mu_ in (r, 0.08):
    S1 = S0 * np.exp((mu_ - 0.5 * sigma**2) + sigma * W_1)
    prod_rows.append({"mu": mu_, "E_disc_S1": float((math.exp(-r) * S1).mean()), "theory": S0 * math.exp(mu_ - r)})
# d(tW) = W dt + t dW: so tW - (integral of W ds) is a martingale; check E[tW - int W] = 0 on paths
paths2, steps2 = 20_000, 500
dW2 = rng.normal(0, math.sqrt(1 / steps2), (paths2, steps2))
W2 = np.cumsum(dW2, axis=1)
intW = (np.concatenate([np.zeros((paths2, 1)), W2[:, :-1]], axis=1) / steps2).sum(axis=1)
out["product"] = {"rows": prod_rows, "E_tW_minus_intW": float((W2[:, -1] - intW).mean()),
                  "var_intW": float(intW.var()), "var_intW_theory": 1 / 3}
out["product"]

## 6. A preview of Girsanov: reweighting shifts the path

Use $Z_T = e^{\theta W_T - \frac12\theta^2T}$ as a weight on the paths (Lesson 12's Radon-Nikodym derivative, now for Brownian motion). Weighted averages behave as if $W$ had been shifted by $\theta t$: $E[Z_Tf(W_T)] = E[f(W_T + \theta T)]$. Lesson 25 turns this into Girsanov's theorem.

In [ ]:
th = 0.5
Zw = np.exp(th * W_1 - 0.5 * th**2)
gir = {"theta": th, "E_ZW": float((Zw * W_1).mean()), "E_ZW_theory": th,
       "E_Z_above": float((Zw * (W_1 > 0.5)).mean()), "shifted_above": 1 - N(0.5 - th), "plain_above": 1 - N(0.5)}
edges = np.linspace(-4, 4, 41)
hz, _ = np.histogram(W_1, bins=edges, weights=Zw, density=True)
h0, _ = np.histogram(W_1, bins=edges, density=True)
mid = (edges[1:] + edges[:-1]) / 2
gir["fig"] = {"mid": mid.tolist(), "plain": h0.tolist(), "weighted": hz.tolist(),
              "ticks": [{"x": x, "label": f"{x:g}"} for x in (-4, -2, 0, 2, 4)]}
out["girsanov"] = gir
{k_: v for k_, v in gir.items() if k_ != "fig"}

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** For each process, find the $dt$ term and decide whether it is a martingale: $W_t^3$, $\;W_t^3 - 3tW_t$, $\;e^{W_t}$, $\;e^{W_t - t/2}$, $\;tW_t$.

**Exercise 2.** Find the constant $c$ that makes $e^{2W_t - ct}$ a martingale. More generally, for which $c$ is $e^{\theta W_t - ct}$ a martingale?

**Exercise 3.** Use the product rule to find $d(tW_t)$, and show that $tW_t - \int_0^tW_s\,ds$ is a martingale. Deduce $E\big[\int_0^1W_s\,ds\big] = 0$.

**Exercise 4.** Show that $W_t^4 - 6tW_t^2 + 3t^2$ is a martingale, and use it to find $E[W_t^4]$.

**Exercise 5.** Let $W$ and $B$ be independent Brownian motions and $X = e^{W_t - t/2}$, $Y = e^{B_t - t/2}$. Use the product rule (with $dW\,dB = 0$) to show $XY$ is a martingale. What changes if $W$ and $B$ have correlation $\rho$?

**Exercise 6.** For $dS = \mu S\,dt + \sigma S\,dW$, use the product rule to find $d(e^{-rt}S_t)$, and show it is a martingale only if $\mu = r$. Compute $E[e^{-r}S_1]$ for $\mu = 8\%$.

**Exercise 7.** (Code) Test the martingale property of $W^3$ and $W^3 - 3tW$ by grouping simulated paths by $W_{0.5}$, as in the lesson. Which one shows a gap, and how does the gap depend on the group?

**Exercise 8.** (Code) Check $E[Z_T\mathbf 1\{W_T > a\}] = P(W_T + \theta T > a)$ for $\theta = 0.5$, $T = 1$ and $a = 0, 0.5, 1$.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")